In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')


import numpy as np
from sklearn.utils import check_random_state
import pandas as pd
from logging import getLogger
from pathlib import Path
import os
import sys
sys.path.append(os.pardir)

import conf
from dataset import generate_real_data
from opl import run_opl

# Set seeds to make the experiment reproducible
random_state = conf.random_state
random_ = check_random_state(random_state)

In [ ]:
logger = getLogger(__name__)
logger.info(f"The current working directory is {Path().cwd()}")

# log path
log_path = Path("../result/num_train")
df_path = log_path / "df"
df_path.mkdir(exist_ok=True, parents=True)

user_data = pd.read_csv("../data/processed_user_feature.csv")
processed_all_data = pd.read_csv("../data/processed_all_data.csv")
action_data = pd.read_csv("../data/processed_unique_action.csv")

In [ ]:
# Initialize the parameters we do not vary in this experiment
x = "num_train"
xticks = conf.num_trains_list
xticklabels = conf.num_trains_list
xlabels = r"logged data size n"

if conf.action_feature == "video_duration":
    features = ["feat", "first_level_category_id", "second_level_category_id", "video_duration"]
elif conf.action_feature == "third_level_category_id":
    features = ["feat", "first_level_category_id", "second_level_category_id", "third_level_category_id"]
elif conf.action_feature == "second_level_category_id":
    features = ["feat", "first_level_category_id", "third_level_category_id", "second_level_category_id"]
elif conf.action_feature == "first_level_category_id":
    features = ["feat", "third_level_category_id", "second_level_category_id", "first_level_category_id"]
elif conf.action_feature == "feat":
    features = ["first_level_category_id", "second_level_category_id","third_level_category_id","feat"]
elif conf.action_feature == "author_id":
    features = ["feat", "first_level_category_id", "second_level_category_id", "third_level_category_id", "author_id"]

grouping_cols = features[:conf.s]
unique_video_ids = action_data.groupby(grouping_cols)["video_id"].first().tolist()

# Calculate 80% of the total number of required video_id
required_size = int(action_data["video_id"].nunique() * 0.8)

if len(unique_video_ids) >= required_size:
    set_of_actions = random_.choice(unique_video_ids, size=required_size, replace=False)
else:
    remaining_video_ids = list(set(action_data["video_id"]) - set(unique_video_ids))
    additional_video_ids = random_.choice(remaining_video_ids, size=required_size - len(unique_video_ids), replace=False)
    set_of_actions = np.concatenate([unique_video_ids, additional_video_ids])

set_of_actions.sort()


dataset_test_all, dataset_test_logging = generate_real_data(
    all_data = processed_all_data,
    user_data = user_data,
    action_data = action_data,
    random_state = conf.random_state, 
    num_data = conf.num_test, 
    dim_feature = conf.dim_feature , 
    num_id_per_each_feature = conf.num_id_per_each_feature, 
    set_of_actions = set_of_actions, 
    beta = conf.beta, 
    s = conf.s,
)

In [ ]:
all_data = pd.DataFrame([], columns = ["seed", x, "method", "V", "existing_action_ratio", "new_action_ratio", "existing_V", "new_V", "per_existing_V", "per_new_V", "estimated_V"])

for num_train in conf.num_trains_list:
    print(f"[num_train={num_train}]")
    
    for seed in range(conf.num_seeds):
        print(f"[{seed}/{conf.num_seeds}]")

        dataset_train_all, dataset_train_logging = generate_real_data(
            all_data = processed_all_data,
            user_data = user_data,
            action_data = action_data,
            random_state = random_state + seed + 1, 
            num_data = num_train, 
            dim_feature = conf.dim_feature , 
            num_id_per_each_feature = conf.num_id_per_each_feature, 
            set_of_actions = set_of_actions, 
            beta = conf.beta, 
            s = conf.s,
        )


        # Calculate the absolute and relative policy values, percentage of the new actions, and the effectiveness of the new actions by the OPL methods
        result_data = run_opl(
            dataset_train_all,
            dataset_test_all,
            dataset_train_logging,
            dataset_test_logging,
            set_of_actions,
            conf.kappa,
            conf.random_state + seed,
        )

        result_data["seed"] = seed
        result_data["num_train"] = num_train
        all_data = pd.concat([all_data, result_data], ignore_index=True)


all_data.to_csv(df_path / "all_data_results.csv", index=False)  
